# AgentCacheBench: M7 Main Experimental Matrix

[![Google Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kshirsagarps/agent-cache-bench/blob/main/notebooks/M7_main_experiments.ipynb)

This notebook executes **Milestone 7 (M7 Main Experimental Matrix)** on Google Colab GPU runtimes.

### Stress Dimensions Evaluated:
- **Context Length Scaling**: $4\text{K} \rightarrow 8\text{K} \rightarrow 16\text{K} \rightarrow 32\text{K}$ tokens
- **Context Mutation Ratios**: $0\%, 5\%, 15\%, 30\%, 50\%$
- **Pause Interruption Delays**: $100\text{ ms}, 1\text{ s}, 5\text{ s}, 30\text{ s}, 300\text{ s}$
- **Memory Pressure Levels**: Low ($<50\%$), Medium ($50-85\%$), High ($>85\%$)

In [ ]:
# Step 1: Environment Setup & Drive Mount
import os, sys, json, torch
if not os.path.exists('agentcachebench'):
    !git clone https://github.com/kshirsagarps/agent-cache-bench.git
    %cd agent-cache-bench

!pip install -q numpy scipy pandas jsonschema pyyaml matplotlib pillow

from agentcachebench.runner.colab_sync import get_colab_gpu_provenance, mount_google_drive, save_colab_checkpoint
from agentcachebench.runner.engine import BenchmarkRunner
from agentcachebench.workloads.tool_use import generate_tool_use_trajectory
from agentcachebench.workloads.coding import generate_coding_trajectory
from agentcachebench.scenarios.mutations import apply_mid_context_replacement, apply_block_shift

drive_mounted = mount_google_drive()
drive_backup_folder = "/content/drive/MyDrive/AgentCacheBench_Results" if drive_mounted else None

gpu_info = get_colab_gpu_provenance()
print(f"Executing M7 Matrix on Host: {gpu_info}")

In [ ]:
# Step 2: Context Scaling Matrix (4K -> 8K -> 16K -> 32K)
runner = BenchmarkRunner(output_dir="results/raw")
context_lengths = [4096, 8192, 16384, 32768]

print("=== Running Context Length Scaling Matrix ===")
for ctx_len in context_lengths:
    traj = generate_coding_trajectory(num_steps=6, base_file_tokens=ctx_len // 2, seed=ctx_len)
    exp_id = f"ACB_M7_ctx_{ctx_len}"
    res = runner.run_experiment(exp_id, "W2_coding", traj, "S1", "B1", {"enable_pause_decay": False, "max_cache_blocks": 4096})
    if drive_backup_folder:
        save_colab_checkpoint(res, exp_id, drive_backup_dir=drive_backup_folder)
    print(f"Executed {exp_id} - Mean TTFT: {res['metrics_summary']['mean_ttft_ms']} ms")

In [ ]:
# Step 3: Context Mutation Matrix (0% -> 5% -> 15% -> 30% -> 50%)
mutation_ratios = [0.0, 0.05, 0.15, 0.30, 0.50]
print("=== Running Context Mutation Matrix ===")
for ratio in mutation_ratios:
    base_traj = generate_coding_trajectory(num_steps=6, seed=int(ratio * 100) + 50)
    mutated_traj = []
    for step in base_traj:
        tokens = step["prompt_tokens"]
        if ratio > 0 and step["step_id"] in [2, 4]:
            tokens = apply_mid_context_replacement(tokens, replace_ratio=ratio)
            if ratio >= 0.15:
                tokens = apply_block_shift(tokens, shift_size=int(ratio * 10))
        mutated_traj.append({"step_id": step["step_id"], "prompt_tokens": tokens, "pause_ms": 1000.0, "event_type": f"mutation_{int(ratio*100)}pct"})
    
    exp_id = f"ACB_M7_mutation_{int(ratio*100)}pct"
    res = runner.run_experiment(exp_id, "W2_coding", mutated_traj, "S3", "B1", {"enable_pause_decay": False})
    if drive_backup_folder:
        save_colab_checkpoint(res, exp_id, drive_backup_dir=drive_backup_folder)
    print(f"Executed {exp_id} - Compute Avoided: {res['metrics_summary']['mean_actual_compute_avoided']:.4f}")

In [ ]:
# Step 4: Pause Interruption Matrix (100ms -> 1s -> 5s -> 30s -> 300s)
pauses_sec = [0.1, 1.0, 5.0, 30.0, 300.0]
print("=== Running Pause Interruption Matrix ===")
for p_sec in pauses_sec:
    traj = generate_tool_use_trajectory(num_steps=6, pause_ms=p_sec * 1000.0, seed=int(p_sec) + 300)
    exp_id = f"ACB_M7_pause_{int(p_sec)}s"
    res = runner.run_experiment(exp_id, "W1_tool_use", traj, "S4", "B3", {"enable_pause_decay": True, "max_cache_blocks": 256})
    if drive_backup_folder:
        save_colab_checkpoint(res, exp_id, drive_backup_dir=drive_backup_folder)
    print(f"Executed {exp_id} - Evictions: {res['metrics_summary']['eviction_count']}")

In [ ]:
print("=== M7 EXPERIMENTAL MATRIX FULLY COMPLETED ON COLAB ===")